### Week 6, Day 3 — Exercise solution

# A fifth context source: Yahoo Finance

The exercise at the end of `3_lab3.ipynb` asks us to browse an MCP marketplace and add another context source to the notebook, using one of the day's four patterns.

I picked the **Yahoo Finance MCP server** from Smithery (https://smithery.ai/server/@Alex2Yang97/yahoo-finance-mcp), one of the most popular finance servers there. It follows the **Integration** pattern from Part 4 of the lab, and it's a natural companion to the lab's market server:

- the lab's market server answers exactly one question — the *current* price of a symbol
- Yahoo Finance adds the context around that number: recent news, price history, financial statements, analyst recommendations

It wraps the free `yfinance` library, so there is **no API key and no signup**. And we launch it with the exact same `uvx --from git+...` technique the lab already used for the Massive server.

In [ ]:
from dotenv import load_dotenv
from agents import Agent, Runner, trace
from agents.mcp import MCPServerStdio, create_static_tool_filter
import os
from datetime import datetime
from IPython.display import Markdown, display

load_dotenv(override=True)

In [ ]:
# On Windows, a stdio MCP server started from a Jupyter kernel writes to a stderr stream with no
# real file descriptor and crashes with io.UnsupportedOperation: fileno. We send the server's
# stderr to the null device so it always has somewhere real to write, which lets every cell below
# use MCPServerStdio exactly as the OpenAI Agents SDK documents it. Mac and Linux are unaffected.
import functools
import subprocess
import agents.mcp.server

agents.mcp.server.stdio_client = functools.partial(agents.mcp.server.stdio_client, errlog=subprocess.DEVNULL)

### Meet the Yahoo Finance server

The server isn't published on PyPI, but `uvx` can install and run it straight from GitHub — the same pattern the lab used for Massive. The first run downloads and builds the package, so we give it a generous timeout; after that it's cached and fast.

In [ ]:
yahoo_params = {
    "command": "uvx",
    "args": ["--from", "git+https://github.com/Alex2Yang97/yahoo-finance-mcp", "yahoo-finance-mcp"],
}

async with MCPServerStdio(params=yahoo_params, client_session_timeout_seconds=180) as server:
    yahoo_tools = await server.list_tools()

[tool.name for tool in yahoo_tools]

### Context engineering: curate the tools

Nine tools! Options chains and holder info are noise for our question, and every tool schema we hand the agent costs context and invites detours. Just like the lab restricted Tavily to `tavily_search`, we filter this server down to the two tools our task needs: news and price history.

In [ ]:
news_and_history = create_static_tool_filter(
    allowed_tool_names=["get_yahoo_finance_news", "get_historical_stock_prices"]
)

### The lab's market server, unchanged

This is the same Part 4 integration from the lab: Massive if you have an API key, otherwise the local simulated-price server from Day 2. One twist: this notebook lives three folders below `6_mcp`, so for the fallback we point `uv run` at the `6_mcp` folder with `--directory`, where the `backend` package lives.

In [ ]:
massive_api_key = os.getenv("MASSIVE_API_KEY")

if massive_api_key:
    market_params = {
        "command": "uvx",
        "args": ["--with", "mcp<2.0.0", "--from", "git+https://github.com/massive-com/mcp_massive@v0.10.0", "mcp_massive"],
        "env": {"MASSIVE_API_KEY": massive_api_key},
    }
else:
    mcp_folder = os.path.abspath(os.path.join(os.getcwd(), "..", "..", ".."))
    market_params = {"command": "uv", "args": ["run", "--directory", mcp_folder, "-m", "backend.market_server"]}

### Two context sources, one agent

Now the payoff. We hand the agent both servers, and ask a question neither can answer alone: the market server knows the latest price, Yahoo Finance knows the month's trend and the news behind it. Watch the trace to see the agent visit both.

In [ ]:
instructions = """You answer questions about the stock market.
Use your price lookup tool for the latest price, and Yahoo Finance for news and price history.
Ground everything you say in what the tools return."""

request = f"""Give me a brief outlook on Amazon (AMZN):
the latest traded price, how the price has moved over the past month, and what recent news says.
Finish with two or three sentences of overall outlook. For context, the current date is {datetime.now().strftime('%Y-%m-%d')}"""

model = "gpt-5.4-mini"

In [ ]:
async with MCPServerStdio(params=market_params, client_session_timeout_seconds=120) as market_server:
    async with MCPServerStdio(params=yahoo_params, client_session_timeout_seconds=180, tool_filter=news_and_history) as yahoo_server:
        agent = Agent(name="market_analyst", instructions=instructions, model=model, mcp_servers=[market_server, yahoo_server])
        with trace("market_analyst"):
            result = await Runner.run(agent, request, max_turns=20)
        display(Markdown(result.final_output))